In [ ]:
from google.colab import drive
drive.mount( '/content/gdrive' )

Mounted at /content/gdrive


In [ ]:
#Các hàm cần thiết để tính toán
import numpy as np
from numpy import pi
import matplotlib.pyplot as plt
import csv 
import cv2
import math

In [ ]:
e = math.e

# %%

def transform(x, y, x_offset, y_offset, angle):
    xr = (x - x_offset) * math.cos(angle * math.pi / 180) - (y - y_offset) * math.sin(angle * math.pi / 180)
    yr = (x - x_offset) * math.sin(angle * math.pi / 180) + (y - y_offset) * math.cos(angle * math.pi / 180)
    return xr, yr


# %%

def calSR1(x, y, z, lc, dc, delta, xx, zz):
    # calSR1 = Exp(zz / sicma) * (z - zz) * (1 / ((x - xx) ^ 2 + (y + l / 2) ^ 2 + (z - zz) ^ 2) ^ 1.5)  ' case 1

    out = e ** (zz / delta) * (z - zz) * (1 / ((x - xx) ** 2 + (y + lc / 2) ** 2 + (z - zz) ** 2) ** 1.5)
    return out


# %%

def calSR2(x, y, z, wc, lc, dc, delta, yy, zz):
    # calSR2 = Exp(zz / sicma) * (z - zz) * ((-2 * yy - l + d) / d) * (1 / ((x + w / 2) ^ 2 + (y - yy) ^ 2 + (z - zz) ^ 2) ^ 1.5 + 1 / ((x - w / 2) ^ 2 + (y - yy) ^ 2 + (z - zz) ^ 2) ^ 1.5) ' case 3

    out = e ** (zz / delta) * (z - zz) * ((-2 * yy - lc + dc) / dc) * (
                1 / ((x + wc / 2) ** 2 + (y - yy) ** 2 + (z - zz) ** 2) ** 1.5 + 1 / (
                    (x - wc / 2) ** 2 + (y - yy) ** 2 + (z - zz) ** 2) ** 1.5)
    return out


# %%

def calSR3(x, y, z, wc, lc, dc, delta, xx, zz):
    # calSR3 = Exp(zz / sicma) * (z - zz) * (1 / ((x - xx) ^ 2 + y ^ 2 + (z - zz) ^ 2) ^ 1.5)  ' case 1

    out = e ** (zz / delta) * (z - zz) * (1 / ((x - xx) ** 2 + y ** 2 + (z - zz) ** 2) ** 1.5)
    return out


# %%

def calSR4(x, y, z, wc, lc, dc, delta, yy, zz):
    # calSR4 = Exp(zz / sicma) * (z - zz) * ((-2 * yy + d) / d) * (1 / ((x + w / 2) ^ 2 + (y - yy) ^ 2 + (z - zz) ^ 2) ^ 1.5 + 1 / ((x - w / 2) ^ 2 + (y - yy) ^ 2 + (z - zz) ^ 2) ^ 1.5) ' case 3

    out = e ** (zz / delta) * (z - zz) * ((-2 * yy + dc) / dc) * (
                1 / ((x + wc / 2) ** 2 + (y - yy) ** 2 + (z - zz) ** 2) ** 1.5 + 1 / (
                    (x - wc / 2) ** 2 + (y - yy) ** 2 + (z - zz) ** 2) ** 1.5)
    return out


# %%

def calSR5(x, y, z, wc, lc, dc, delta, xx, zz):
    # calSR5 = Exp(zz / sicma) * (z - zz) * (1 / ((x - xx) ^ 2 + (y - l / 2) ^ 2 + (z - zz) ^ 2) ^ 1.5)   ' case 1

    out = e ** (zz / delta) * (z - zz) * (1 / ((x - xx) ** 2 + (y - lc / 2) ** 2 + (z - zz) ** 2) ** 1.5)
    return out


# %%

def calSR6(x, y, z, wc, lc, dc, delta, yy, zz):
    # calSR6 = Exp(zz / sicma) * (z - zz) * ((yy - l / 2 + d) / d) * (1 / ((x + w / 2) ^ 2 + (y - yy) ^ 2 + (z - zz) ^ 2) ^ 1.5 + 1 / ((x - w / 2) ^ 2 + (y - yy) ^ 2 + (z - zz) ^ 2) ^ 1.5) ' case 3

    out = e ** (zz / delta) * (z - zz) * ((yy - lc / 2 + dc) / dc) * (
                1 / ((x + wc / 2) ** 2 + (y - yy) ** 2 + (z - zz) ** 2) ** 1.5 + 1 / (
                    (x - wc / 2) ** 2 + (y - yy) ** 2 + (z - zz) ** 2) ** 1.5)
    return out


# %%

def InteSR1(x, y, z, wc, lc, dc, delta):
    n = 20
    m = 20
    a = -wc / 2
    b = wc / 2
    c = -dc / 2
    d = 0

    h = (d - c) / n
    k = (b - a) / m

    T2 = 0
    T3 = 0
    T4 = 0

    T1 = calSR1(x, y, z, lc, dc, delta, a, c) + calSR1(x, y, z, lc, dc, delta, b, c) + calSR1(x, y, z, lc, dc, delta, a,
                                                                                              d) + calSR1(x, y, z, lc,
                                                                                                          dc, delta, b,
                                                                                                          d)

    for j in range(1, n):
        yj = c + j * h
        T3 = T3 + calSR1(x, y, z, lc, dc, delta, a, yj) + calSR1(x, y, z, lc, dc, delta, b, yj)

    for i in range(1, m):
        xi = a + i * k
        T2 = T2 + calSR1(x, y, z, lc, dc, delta, xi, c) + calSR1(x, y, z, lc, dc, delta, xi, d)
        for j in range(1, n):
            yj = c + j * h
            T4 = T4 + calSR1(x, y, z, lc, dc, delta, xi, yj)

    out = h * k * (T1 / 4 + T2 / 2 + T3 / 2 + T4)

    return out


# %%

def InteSR2(x, y, z, wc, lc, dc, delta):
    n = 20
    m = 20

    a = -lc / 2
    b = (-lc + dc) / 2
    c = -dc / 2
    d = 0

    h = (d - c) / n
    k = (b - a) / m

    T2 = 0
    T3 = 0
    T4 = 0

    T1 = calSR2(x, y, z, wc, lc, dc, delta, a, c) + calSR2(x, y, z, wc, lc, dc, delta, b, c) + calSR2(x, y, z, wc, lc,
                                                                                                      dc, delta, a,
                                                                                                      d) + calSR2(x, y,
                                                                                                                  z, wc,
                                                                                                                  lc,
                                                                                                                  dc,
                                                                                                                  delta,
                                                                                                                  b, d)

    for j in range(1, n):
        yj = c + j * h
        T3 = T3 + calSR2(x, y, z, wc, lc, dc, delta, a, yj) + calSR2(x, y, z, wc, lc, dc, delta, b, yj)

    for i in range(1, m):
        xi = a + i * k
        T2 = T2 + calSR2(x, y, z, wc, lc, dc, delta, xi, c) + calSR2(x, y, z, wc, lc, dc, delta, xi, d)
        for j in range(1, n):
            yj = c + j * h
            T4 = T4 + calSR2(x, y, z, wc, lc, dc, delta, xi, yj)

    out = h * k * (T1 / 4 + T2 / 2 + T3 / 2 + T4)

    return out


# %%

def InteSR3(x, y, z, wc, lc, dc, delta):
    n = 20
    m = 20

    a = -wc / 2
    b = wc / 2
    c = -dc
    d = -dc / 2

    h = (d - c) / n
    k = (b - a) / m

    T2 = 0
    T3 = 0
    T4 = 0

    T1 = calSR3(x, y, z, wc, lc, dc, delta, a, c) + calSR3(x, y, z, wc, lc, dc, delta, b, c) + calSR3(x, y, z, wc, lc,
                                                                                                      dc, delta, a,
                                                                                                      d) + calSR3(x, y,
                                                                                                                  z, wc,
                                                                                                                  lc,
                                                                                                                  dc,
                                                                                                                  delta,
                                                                                                                  b, d)

    for j in range(1, n):
        yj = c + j * h
        T3 = T3 + calSR3(x, y, z, wc, lc, dc, delta, a, yj) + calSR3(x, y, z, wc, lc, dc, delta, b, yj)

    for i in range(1, m):
        xi = a + i * k
        T2 = T2 + calSR3(x, y, z, wc, lc, dc, delta, xi, c) + calSR3(x, y, z, wc, lc, dc, delta, xi, d)
        for j in range(1, n):
            yj = c + j * h
            T4 = T4 + calSR3(x, y, z, wc, lc, dc, delta, xi, yj)

    out = h * k * (T1 / 4 + T2 / 2 + T3 / 2 + T4)

    return out


# %%

def InteSR4(x, y, z, wc, lc, dc, delta):
    n = 20
    m = 20

    a = 0
    b = dc / 2
    c = -dc
    d = -dc / 2

    h = (d - c) / n
    k = (b - a) / m

    T2 = 0
    T3 = 0
    T4 = 0

    T1 = calSR4(x, y, z, wc, lc, dc, delta, a, c) + calSR4(x, y, z, wc, lc, dc, delta, b, c) + calSR4(x, y, z, wc, lc,
                                                                                                      dc, delta, a,
                                                                                                      d) + calSR4(x, y,
                                                                                                                  z, wc,
                                                                                                                  lc,
                                                                                                                  dc,
                                                                                                                  delta,
                                                                                                                  b, d)

    for j in range(1, n):
        yj = c + j * h
        T3 = T3 + calSR4(x, y, z, wc, lc, dc, delta, a, yj) + calSR4(x, y, z, wc, lc, dc, delta, b, yj)

    for i in range(1, m):
        xi = a + i * k
        T2 = T2 + calSR4(x, y, z, wc, lc, dc, delta, xi, c) + calSR4(x, y, z, wc, lc, dc, delta, xi, d)
        for j in range(1, n):
            yj = c + j * h
            T4 = T4 + calSR4(x, y, z, wc, lc, dc, delta, xi, yj)

    out = h * k * (T1 / 4 + T2 / 2 + T3 / 2 + T4)

    return out


# %%

def InteSR5(x, y, z, wc, lc, dc, delta):
    n = 20
    m = 20

    a = -wc / 2
    b = wc / 2
    c = -dc
    d = 0

    h = (d - c) / n
    k = (b - a) / m

    T2 = 0
    T3 = 0
    T4 = 0

    T1 = calSR5(x, y, z, wc, lc, dc, delta, a, c) + calSR5(x, y, z, wc, lc, dc, delta, b, c) + calSR5(x, y, z, wc, lc,
                                                                                                      dc, delta, a,
                                                                                                      d) + calSR5(x, y,
                                                                                                                  z, wc,
                                                                                                                  lc,
                                                                                                                  dc,
                                                                                                                  delta,
                                                                                                                  b, d)

    for j in range(1, n):
        yj = c + j * h
        T3 = T3 + calSR5(x, y, z, wc, lc, dc, delta, a, yj) + calSR5(x, y, z, wc, lc, dc, delta, b, yj)

    for i in range(1, m):
        xi = a + i * k
        T2 = T2 + calSR5(x, y, z, wc, lc, dc, delta, xi, c) + calSR5(x, y, z, wc, lc, dc, delta, xi, d)
        for j in range(1, n):
            yj = c + j * h
            T4 = T4 + calSR5(x, y, z, wc, lc, dc, delta, xi, yj)

    out = h * k * (T1 / 4 + T2 / 2 + T3 / 2 + T4)

    return out


# %%

def InteSR6(x, y, z, wc, lc, dc, delta):
    n = 20
    m = 20

    a = lc / 2 - dc
    b = lc / 2
    c = -dc
    d = 0

    h = (d - c) / n
    k = (b - a) / m

    T2 = 0
    T3 = 0
    T4 = 0

    T1 = calSR6(x, y, z, wc, lc, dc, delta, a, c) + calSR6(x, y, z, wc, lc, dc, delta, b, c) + calSR6(x, y, z, wc, lc,
                                                                                                      dc, delta, a,
                                                                                                      d) + calSR6(x, y,
                                                                                                                  z, wc,
                                                                                                                  lc,
                                                                                                                  dc,
                                                                                                                  delta,
                                                                                                                  b, d)

    for j in range(1, n):
        yj = c + j * h
        T3 = T3 + calSR6(x, y, z, wc, lc, dc, delta, a, yj) + calSR6(x, y, z, wc, lc, dc, delta, b, yj)

    for i in range(1, m):
        xi = a + i * k
        T2 = T2 + calSR6(x, y, z, wc, lc, dc, delta, xi, c) + calSR6(x, y, z, wc, lc, dc, delta, xi, d)
        for j in range(1, n):
            yj = c + j * h
            T4 = T4 + calSR6(x, y, z, wc, lc, dc, delta, xi, yj)

    out = h * k * (T1 / 4 + T2 / 2 + T3 / 2 + T4)

    return out


# %%

def calTST(x, y, z, lc, dc, delta, xx, zz):
    # calTST = Exp(zz / sicma) * (z - zz) * (1 / ((x - xx) ^ 2 + (y + l / 3 / d * zz + l / 3) ^ 2 + (z - zz) ^ 2) ^ 1.5)

    out = e ** (zz / delta) * (z - zz) * (
                1 / ((x - xx) ** 2 + (y + lc / 3 / dc * zz + lc / 3) ** 2 + (z - zz) ** 2) ** 1.5)
    return out


# %%

def InteTST(x, y, z, wc, lc, dc, delta):
    n = 20
    m = 20

    a = -wc / 2
    b = wc / 2
    c = -dc
    d = 0

    h = (d - c) / n
    k = (b - a) / m

    T2 = 0
    T3 = 0
    T4 = 0

    T1 = calTST(x, y, z, lc, dc, delta, a, c) + calTST(x, y, z, lc, dc, delta, b, c) + calTST(x, y, z, lc, dc, delta, a,
                                                                                              d) + calTST(x, y, z, lc,
                                                                                                          dc, delta, b,
                                                                                                          d)

    for j in range(1, n):
        yj = c + j * h
        T3 = T3 + calTST(x, y, z, lc, dc, delta, a, yj) + calTST(x, y, z, lc, dc, delta, b, yj)

    for i in range(1, m):
        xi = a + i * k
        T2 = T2 + calTST(x, y, z, lc, dc, delta, xi, c) + calTST(x, y, z, lc, dc, delta, xi, d)
        for j in range(1, n):
            yj = c + j * h
            T4 = T4 + calTST(x, y, z, lc, dc, delta, xi, yj)

    out = h * k * (T1 / 4 + T2 / 2 + T3 / 2 + T4)

    return out


# %%

def calT_add3(x, y, z, wc, lc, dc, delta, yy, zz):
    # calT_add3 = (-2 * yy / l) * Exp(zz / sicma) * (z - zz) * (1 / ((x + w / 2) ^ 2 + (y - yy) ^ 2 + (z - zz) ^ 2) ^ 1.5 + 1 / ((x - w / 2) ^ 2 + (y - yy) ^ 2 + (z - zz) ^ 2) ^ 1.5) ' case 3

    out = (-2 * yy / lc) * e ** (zz / delta) * (z - zz) * (
                1 / ((x + wc / 2) ** 2 + (y - yy) ** 2 + (z - zz) ** 2) ** 1.5 + 1 / (
                    (x - wc / 2) ** 2 + (y - yy) ** 2 + (z - zz) ** 2) ** 1.5)
    return out


# %%

def InteT_add3(x, y, z, wc, lc, dc, delta):
    n = 20
    m = 20

    x_ = np.zeros(m + 1)
    y_ = np.zeros((n + 1, m + 1))
    c = np.zeros(m + 1)
    d = np.zeros(m + 1)
    h = np.zeros(m + 1)

    # sinalfa = dc / ((dc ^ 2 + (lc / 2) ^ 2) ^ 0.5)
    sinalfa = dc / ((dc ** 2 + (lc / 2) ** 2) ** 0.5)

    a = -lc / 2
    b = 0

    k = (b - a) / m

    T2 = 0
    T3 = 0
    T4 = 0

    for j in range(m + 1):
        x_[j] = a + j * k
        c[j] = -x_[j] * 2 * dc / lc - dc
        d[j] = 0
        h[j] = (d[j] - c[j]) / n
        for i in range(n + 1):
            y_[i, j] = c[j] + i * h[j]

    T1 = h[0] * (calT_add3(x, y, z, wc, lc, dc, delta, x_[0], c[0]) + calT_add3(x, y, z, wc, lc, dc, delta, x_[0],
                                                                                d[0])) + h[m] * (
                     calT_add3(x, y, z, wc, lc, dc, delta, x_[m], c[m]) + calT_add3(x, y, z, wc, lc, dc, delta, x_[m],
                                                                                    d[m]))

    for i in range(n):
        T3 = T3 + h[0] * calT_add3(x, y, z, wc, lc, dc, delta, x_[0], y_[i, 0]) + h[m] * calT_add3(x, y, z, wc, lc, dc,
                                                                                                   delta, x_[m],
                                                                                                   y_[i, m])

    for j in range(m):

        T2 = T2 + (calT_add3(x, y, z, wc, lc, dc, delta, x_[j], c[j]) + calT_add3(x, y, z, wc, lc, dc, delta, x_[j],
                                                                                  d[j])) * h[j]

        for i in range(n):
            T4 = T4 + (calT_add3(x, y, z, wc, lc, dc, delta, x_[j], y_[i, j])) * h[j]

    T1 = k * T1 / 4
    T2 = k * T2 / 2
    T3 = k * T3 / 2
    T4 = k * T4

    out = sinalfa * (T1 + T2 + T3 + T4)
    return out


# %%
# Ham Tong Quat

def MagneSR(flag, x, y, z, wc, lc, dc, delta):
    if flag == 1:  # Step R
        out = InteSR1(x, y, z, wc, lc, dc, delta) + InteSR2(x, y, z, wc, lc, dc, delta) + InteSR3(x, y, z, wc, lc, dc, delta) + InteSR4(x, y, z, wc, lc, dc, delta) - InteSR5(x, y, z, wc, lc, dc, delta) - InteSR6(x, y, z, wc, lc, dc, delta)
    else:  # Step T
        out = InteSR1(x, y, z, wc, lc, dc, delta) + InteSR2(x, y, z, wc, lc, dc, delta) + InteT_add3(x, y - lc / 6, z, wc, 2 * lc / 3, dc / 2, delta) + InteTST(x, y, z, wc, lc, dc, delta) - InteSR5(x, y, z, wc, lc, dc, delta) - InteSR6(x, y, z, wc, lc, dc, delta)
    return out

In [ ]:
wc_all = np.linspace(0.3,1.5,12)
lc_all = np.linspace(2,20,19)
dc_all = np.linspace(0.1,3,29)

In [ ]:
import os
import csv
path1 = "/content/gdrive/MyDrive/Dipole Model/Data/Low Data"

In [ ]:
N = 16
Res = 0.78  # Resolution mm
aR = np.linspace(-N * Res, N * Res, 2 * N )
bR = np.linspace(-N * Res, N * Res, 2 * N )
H = np.zeros((len(aR), len(bR)))
x_offset=0
y_offset=0
angle=0
wc=0.5
lc=10
dc=3
z=1
freq=5000
sicma=35461000
mu=0.0000012566
csi=0.085
K=1.5
I=0.01
G=3981

delta = 1 / np.sqrt(pi * freq * mu * sicma) * 1000
flag=0

for wc in wc_all:
  for lc in lc_all:
    for dc in dc_all:
      tam = 'Step_T' + '_' + str(wc) + '_' + str(lc) + '_' + str(dc) + '.csv'
      if(tam not in  os.listdir(os.path.expanduser(path1))):
        print(tam)
        H = np.zeros((len(aR), len(bR)))
        for u, xo in enumerate(aR):
          for v, yo in enumerate(aR):
            x, y = transform(xo, yo, x_offset, y_offset, angle)
            H[u, v] = MagneSR(flag, x, y, z, wc, lc, dc, delta)
            H[u, v] = math.cos(angle * math.pi / 180) * H[u, v]

        H = csi/4/pi*H
        H = K*I*G*H
        name_params=['N','liff_off','Csi','K','I','f','sicma','mu','G','Res']
        params=[32, z, csi, K, I, freq, sicma, mu, G, Res]
        L = ['shape', 'width', 'length', 'depth','angle','x_offset','y_offset']
        C = ['Step_T', wc, lc, dc, angle, x_offset, y_offset]

        filename = '/content/gdrive/MyDrive/Dipole Model/Data/Low Data/' + C[0] + '_' + str(C[1]) + '_' + str(C[2]) + '_' + str(C[3]) + '.csv'

        with open(filename, 'w', newline='') as f:
          writer = csv.writer(f)
          writer.writerow(name_params)
          writer.writerow(params)
          writer.writerow(L)
          writer.writerow(C)
          writer.writerows(H)
          f.close()

Step_T_1.3909090909090909_11.0_2.585714285714286.csv
Step_T_1.3909090909090909_11.0_2.689285714285714.csv
Step_T_1.3909090909090909_11.0_2.7928571428571427.csv
Step_T_1.3909090909090909_11.0_2.8964285714285714.csv
Step_T_1.3909090909090909_11.0_3.0.csv
Step_T_1.3909090909090909_12.0_0.1.csv
Step_T_1.3909090909090909_12.0_0.20357142857142857.csv
Step_T_1.3909090909090909_12.0_0.30714285714285716.csv
Step_T_1.3909090909090909_12.0_0.4107142857142857.csv
Step_T_1.3909090909090909_12.0_0.5142857142857142.csv
Step_T_1.3909090909090909_12.0_0.6178571428571428.csv
Step_T_1.3909090909090909_12.0_0.7214285714285714.csv
Step_T_1.3909090909090909_12.0_0.825.csv
Step_T_1.3909090909090909_12.0_0.9285714285714285.csv
Step_T_1.3909090909090909_12.0_1.032142857142857.csv
Step_T_1.3909090909090909_12.0_1.1357142857142857.csv
Step_T_1.3909090909090909_12.0_1.2392857142857143.csv
Step_T_1.3909090909090909_12.0_1.342857142857143.csv
Step_T_1.3909090909090909_12.0_1.4464285714285714.csv
Step_T_1.3909090909